# 24｜附錄A_自動微分與計算圖

以中央差分檢查解析梯度，並觀察步長過大或過小的誤差。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

## 投影片與程式的閱讀方式

對應 `slides/24_附錄A_自動微分與計算圖.md`，目前 21 頁。逐圖完整表見 `programs/SLIDE_NOTEBOOK_MAP_05_25.md`。

對偶數、可執行反向引擎、計算圖、分支累積與 detach。

各格會標示實驗資料、評估方式與適用範圍；架構圖用於說明機制與張量形狀。

| 程式代碼 | 投影片頁次 | 本格學習內容 |
|---|---|---|
| 24-03 | 4、5、6、7 | 課堂小實驗 |
| 24-04 | 3、6、7 | 執行結果圖 |
| 24-05 | 4、5、8、9、10、11、14、18、21 | 1. 前向模式與對偶數：一次傳一個方向 |
| 24-06 | 8、9、12、13、14、15、16、17、18、19、20、21 | 2. 小型反向自動微分引擎 |

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
# 程式 24-01｜投影片 24：2 頁
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
# 程式 24-02｜投影片 24：2 頁
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    svg_path = folder / f"{name}.svg"
    plt.gcf().savefig(svg_path, bbox_inches="tight", metadata={"Date": None})
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines()) + "\n")
    plt.show()
    plt.close(plt.gcf())

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

## 課堂小實驗

In [ ]:
# 程式 24-03｜投影片 24：4、5、6、7 頁
def f(x, y): return x*x*y + y + 2
x, y, check_eps = 3.0, 4.0, 1e-5
analytic = np.array([2*x*y, x*x+1])
numeric_f = np.array([(f(x+check_eps,y)-f(x-check_eps,y))/(2*check_eps),
                      (f(x,y+check_eps)-f(x,y-check_eps))/(2*check_eps)])

# f 對 x 是二次式，中央差分會特別精準；另用非多項式 g 觀察步長誤差。
def g(v): return np.sin(v) * np.exp(v/10)
g_prime = np.exp(x/10) * (np.cos(x) + 0.1*np.sin(x))
eps_values = np.logspace(-1, -14, 14)
numeric_g = np.array([(g(x+e)-g(x-e))/(2*e) for e in eps_values])
errors = np.abs(numeric_g-g_prime)
best = int(np.argmin(errors))
print('f analytic gradient:', analytic, ' finite difference:', numeric_f)
print('g best epsilon:', eps_values[best], 'error:', errors[best])

## 執行結果圖

In [ ]:
# 程式 24-04｜投影片 24：3、6、7 頁
plt.loglog(eps_values, errors, 'o-')
plt.axvline(eps_values[best], color='tab:red', ls='--', label=f'best eps={eps_values[best]:.0e}')
plt.gca().invert_xaxis(); plt.xlabel('epsilon'); plt.ylabel('gradient error')
plt.title('Finite differences fail when epsilon is too large or too small'); plt.legend()
savefig('24_autodiff_demo')
report('autodiff', {'f_analytic': analytic.tolist(), 'f_numeric': numeric_f.tolist(),
                    'g_best_epsilon': eps_values[best], 'g_best_error': errors[best]})

## 1. 前向模式與對偶數：一次傳一個方向

加法與乘法保存數值和方向導數。把 x 的 tangent 設為 1、y 設為 0 可求 ∂f/∂x；交換種子就求另一個偏導。這不是有限差分。

In [ ]:
# 程式 24-05｜投影片 24：4、5、8、9、10、11、14、18、21 頁
class Dual24:
    def __init__(self,value,tangent=0.):self.value=float(value);self.tangent=float(tangent)
    def __add__(self,other):
        other=other if isinstance(other,Dual24) else Dual24(other)
        return Dual24(self.value+other.value,self.tangent+other.tangent)
    __radd__=__add__
    def __mul__(self,other):
        other=other if isinstance(other,Dual24) else Dual24(other)
        return Dual24(self.value*other.value,self.tangent*other.value+self.value*other.tangent)
    __rmul__=__mul__
rows24=[]
for seed24 in [(1.,0.),(0.,1.)]:
    xd24=Dual24(3,seed24[0]);yd24=Dual24(4,seed24[1]);ad24=xd24*xd24;bd24=ad24*yd24;cd24=bd24+yd24;fd24=cd24+2
    rows24.append([seed24,ad24.tangent,bd24.tangent,cd24.tangent,fd24.tangent])
    assert fd24.value==42
assert rows24[0][-1]==24 and rows24[1][-1]==10
print(pd.DataFrame(rows24,columns=['seed (dx,dy)','da','db','dc','df']))
# 方向 (2,-1) 的 JVP 等於 grad dot direction
mixed24=f(Dual24(3,2),Dual24(4,-1));assert mixed24.tangent==38

## 2. 小型反向自動微分引擎

建立運算節點、拓樸排序，再從輸出反向套用局部導數。重複父節點 x*x 必須累加兩次。此簡化引擎保留圖；不模擬 PyTorch 的釋放圖或版本檢查。

In [ ]:
# 程式 24-06｜投影片 24：8、9、12、13、14、15、16、17、18、19、20、21 頁
class Value24:
    def __init__(self,value,parents=(),name=''):
        self.value=float(value);self.parents=parents;self.grad=0.;self.name=name;self.local=lambda:None
    def __add__(self,other):
        other=other if isinstance(other,Value24) else Value24(other,name=str(other));out=Value24(self.value+other.value,(self,other),'+')
        def backward():self.grad+=out.grad;other.grad+=out.grad
        out.local=backward;return out
    __radd__=__add__
    def __mul__(self,other):
        other=other if isinstance(other,Value24) else Value24(other,name=str(other));out=Value24(self.value*other.value,(self,other),'*')
        def backward():self.grad+=other.value*out.grad;other.grad+=self.value*out.grad
        out.local=backward;return out
    __rmul__=__mul__
    def detach(self):return Value24(self.value,name='detached')
    def backward(self):
        seen=set();order=[]
        def visit(v):
            if id(v) in seen:return
            seen.add(id(v))
            for p in v.parents:visit(p)
            order.append(v)
        visit(self)
        for v in order:
            if v.parents:v.grad=0.  # 中間梯度重算；葉節點保留累積
        self.grad=1.
        for v in reversed(order):v.local()
        return order
xv24,yv24=Value24(3,name='x'),Value24(4,name='y');av24=xv24*xv24;bv24=av24*yv24;cv24=bv24+yv24;fv24=cv24+2
nodes24=fv24.backward();assert [fv24.value,xv24.grad,yv24.grad]==[42,24,10]
print(pd.DataFrame([{'node':v.name,'value':v.value,'adjoint':v.grad} for v in nodes24]))
fv24.backward();assert [xv24.grad,yv24.grad]==[48,20]
xv24.grad=yv24.grad=0.;fv24.backward();assert [xv24.grad,yv24.grad]==[24,10]
xd24,yd24=Value24(3,name='x'),Value24(4,name='y');detached24=(xd24*xd24).detach()*yd24+yd24+2;detached24.backward()
assert xd24.grad==0 and yd24.grad==10
print('Detached x branch: custom x.grad=0, y.grad=10; PyTorch disconnected leaf would be None')
# 同一計算圖，前向標 primal；反向標 adjoint
from matplotlib.patches import FancyArrowPatch
coords24={'x':(0,1),'y':(0,-1),'a':(1,1),'b':(2,0),'c':(3,0),'2':(3,-1),'f':(4,0)}
values24={'x':3,'y':4,'a':9,'b':36,'c':40,'2':2,'f':42};grads24={'x':24,'y':10,'a':4,'b':1,'c':1,'2':1,'f':1}
edges24=[('x','a'),('x','a'),('a','b'),('y','b'),('b','c'),('y','c'),('c','f'),('2','f')]
fig,axes=plt.subplots(2,1,figsize=(11,6))
for ax,display24,title24 in zip(axes,[values24,grads24],['Forward values','Reverse accumulated derivatives']):
    for j,(start,end) in enumerate(edges24):
        p,q=coords24[start],coords24[end]
        if ax is axes[1]:p,q=q,p
        ax.add_patch(FancyArrowPatch(p,q,arrowstyle='->',mutation_scale=14,shrinkA=18,shrinkB=18,connectionstyle=f'arc3,rad={.18 if j==1 else 0}',color='gray'))
    for name,(px,py) in coords24.items():ax.text(px,py,f'{name}\n{display24[name]}',ha='center',va='center',bbox={'boxstyle':'round','facecolor':'white','edgecolor':'steelblue'})
    ax.set(xlim=(-.4,4.4),ylim=(-1.5,1.5),title=title24);ax.axis('off')
savefig('c24_computation_graph')

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。